# D1000 CPU evaluation recovery v1
Evaluation only: no training, inference or generation. Keep this notebook private. Kaggle remains the sole persistent research store.


## Required human setup
Select CPU (no accelerator), then configure Colab Secrets `KAGGLE_API_TOKEN`, `GROQ_API_KEY`, `GEMINI_API_KEY` and enable this notebook. Never enter credentials in code or chat. Confirm your project permits private checkpoint processing in your own Colab VM before enabling the transfer gate. If permission or any secret is missing, stop here. See the accompanying recovery README for the full protocol.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.2.5', 'groq==1.7.0', 'tiktoken==0.14.0', 'matplotlib', 'jsonschema'], check=True)
recovery_source = '"""Colab CPU adapter v1 for the exact supported D1000 evaluation source.\n\nNo research code is changed on disk. A narrowly checked restore adapter removes\nKaggle Secrets loading and the read-only input path; credentials use Colab userdata.\n"""\nimport ast\nfrom contextlib import redirect_stdout, redirect_stderr\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport tempfile\nimport uuid\n\nSTORE = \'scarletthe0116/uk-motor-d1000-models-results\'\nKERNEL = \'scarletthe0116/uk-motor-d1000-train-eval\'\nSOURCE_SHA = \'35e54432a06d06285699d8e9096cabd1bf2811693515934a092b1ea2c09d7133\'\nARCHIVE_SHA = \'aa9e5eba75d908c13da03c0c64ce47c55794e9aaeb74d16a25312fd90f86a604\'\nINPUT_ROOT = Path(\'/content/d1000-private-input\')\nSCRATCH = \'scarletthe0116/uk-motor-d1000-recovery-scratch-v1\'\nLEASE = \'scarletthe0116/uk-motor-d1000-evaluation-lease-v1\'\nTERMINAL = {\'COMPLETE\', \'ERROR\', \'CANCEL_ACKNOWLEDGED\'}\nCREDENTIAL_PROLOGUE = \'\'\'    from kaggle import api\n    from kaggle_secrets import UserSecretsClient\n    import os\n    secrets=UserSecretsClient()\n    for key in (\'KAGGLE_API_TOKEN\',\'GROQ_API_KEY\',\'GEMINI_API_KEY\'):\n        os.environ[key]=secrets.get_secret(key)\n    api.authenticate()\n\'\'\'\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\n\ndef supported_bundle(notebook):\n    """Read exact private code snapshot, not arbitrary downloaded executable code."""\n    source = \'\'.join(\'\'.join(c[\'source\']) for c in notebook[\'cells\'] if c[\'cell_type\'] == \'code\')\n    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_SHA:\n        raise ValueError(\'Supported source hash mismatch\')\n    assignments = [n for n in ast.parse(source).body if isinstance(n, ast.Assign)\n                   and any(isinstance(t, ast.Name) and t.id == \'files\' for t in n.targets)]\n    if len(assignments) != 1: raise ValueError(\'Expected one frozen code bundle\')\n    files = ast.literal_eval(assignments[0].value)\n    for name, content in files.items():\n        p = PurePosixPath(name)\n        if p.is_absolute() or \'..\' in p.parts or not isinstance(content, str):\n            raise ValueError(\'Unsafe source bundle path\')\n        if p.parts[0] not in (\'experiments\', \'src\', \'eval\', \'cloud\', \'configs\', \'scripts\'):\n            raise ValueError(\'Unexpected source bundle path\')\n    return files\n\n\ndef stage_bundle(code, files):\n    """Reuse verified code after a preflight failure, never overwrite changed code."""\n    code = Path(code); code.mkdir(parents=True, exist_ok=True)\n    expected_py = {str(PurePosixPath(n)) for n in files if n.endswith(\'.py\')}\n    actual_py = {str(p.relative_to(code)) for p in code.rglob(\'*.py\')}\n    if actual_py - expected_py:\n        raise RuntimeError(\'Unexpected executable source in recovery directory\')\n    for name, text in files.items():\n        p = code/name\n        if p.exists() and (not p.is_file() or p.is_symlink() or p.read_text() != text):\n            raise RuntimeError(\'Existing recovery source differs from pinned bundle\')\n    for name, text in files.items():\n        p = code/name; p.parent.mkdir(parents=True, exist_ok=True)\n        if not p.exists(): p.write_text(text)\n        os.chmod(p,0o600)\n\n\ndef restore_adapter(source):\n    """Change credential acquisition only; all original restore checks retained."""\n    tree = ast.parse(source)\n    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'restore\')\n    original = ast.get_source_segment(source, node) + \'\\n\'\n    if original.count(CREDENTIAL_PROLOGUE) != 1:\n        raise ValueError(\'Credential prologue differs from reviewed source\')\n    adapted = original.replace(CREDENTIAL_PROLOGUE,\n                               \'    from kaggle import api\\n    api.authenticate()\\n\', 1)\n    # Colab exposes /kaggle/input as read-only; use a private cloud directory.\n    if adapted.count("Path(\'/kaggle/input\')") != 1:\n        raise ValueError(\'Mounted-input expression differs from reviewed source\')\n    adapted = adapted.replace("Path(\'/kaggle/input\')", "Path(\'/content/d1000-private-input\')", 1)\n    # Only this function is compiled, under the unchanged module globals.\n    return compile(adapted, \'<reviewed-colab-restore-v1>\', \'exec\')\n\n\ndef rows(path):\n    return [json.loads(s) for s in Path(path).read_text().splitlines() if s.strip()]\n\n\ndef verify_journals(root):\n    """Validate saved counts and retain exact unknown-outcome protection."""\n    root = Path(root); votes = {}\n    for provider, count in ((\'groq\', 36), (\'gemini\', 345)):\n        entries = rows(root/\'judging\'/f\'{provider}.jsonl\')\n        ids = [r[\'evaluation_id\'] for r in entries]\n        if len(entries) != count or len(set(ids)) != count:\n            raise ValueError(\'Saved judgement population mismatch\')\n        votes[provider] = set(ids)\n    if len(votes[\'groq\'] & votes[\'gemini\']) != 32:\n        raise ValueError(\'Paired population mismatch\')\n    uncertain = {\'groq\': 0, \'gemini\': 0}\n    for provider in uncertain:\n        folder = root/\'judge_runtime/evaluation_cache_v2\'/provider\n        for path in folder.glob(\'*.attempts.json\'):\n            history = json.loads(path.read_text())\n            if not history or history[-1][\'status\'] not in (\'in_flight\',\'transport_uncertain\',\'response_saved\'):\n                continue\n            last = history[-1]; uncertain[provider] += 1\n            if provider == \'gemini\':\n                if last[\'status\'] != \'transport_uncertain\' or last.get(\'error_category\') != \'gemini HTTP 503\':\n                    raise ValueError(\'Unknown-outcome population changed\')\n                if (folder/(last[\'request_sha256\']+\'.raw.json\')).exists():\n                    raise ValueError(\'New durable reply requires separate reconciliation\')\n    if uncertain != {\'groq\': 0, \'gemini\': 1}:\n        raise ValueError(\'Expected protected Gemini uncertain request\')\n    # Hash all journals, caches and quota ledgers; this function never edits them.\n    retained = {str(p.relative_to(root)): sha(p) for directory in (\'judging\', \'judge_runtime\')\n                for p in (root/directory).rglob(\'*\') if p.is_file()}\n    return {\'groq\':36, \'gemini\':345, \'paired\':32, \'uncertain\':uncertain,\n            \'journals_manifest_sha256\':hashlib.sha256(json.dumps(retained,sort_keys=True).encode()).hexdigest()}\n\n\nclass ColabRecovery:\n    def __init__(self):\n        self.ready = False; self.started = False\n        self.session = uuid.uuid4().hex\n        self.receipt = {\'adapter\':\'colab_cpu_v1_1\',\'status\':\'BLOCKED\',\'new_judge_calls\':0}\n\n    def prepare(self, *, private_colab_transfer_approved=False):\n        # This explicit gate comes before credential retrieval or private downloads.\n        if not private_colab_transfer_approved:\n            raise PermissionError(\'Confirm project permission for private checkpoint processing in your Colab VM\')\n        if not Path(\'/content\').is_dir(): raise RuntimeError(\'Managed Colab runtime required\')\n        from google.colab import userdata\n        if os.environ.get(\'COLAB_GPU\') not in (None, \'\', \'0\'):\n            raise RuntimeError(\'Select CPU runtime first\')\n        Path(\'/kaggle/working\').mkdir(parents=True, exist_ok=True)\n        os.chmod(\'/kaggle/working\', 0o700)\n        self.log = Path(\'/kaggle/working/recovery-private.log\')\n        self.log.touch(mode=0o600, exist_ok=True); os.chmod(self.log,0o600)\n        # Colab Secrets are local to this notebook/account; no Kaggle key transfer.\n        for key in (\'KAGGLE_API_TOKEN\', \'GROQ_API_KEY\', \'GEMINI_API_KEY\'):\n            value = userdata.get(key)\n            if not isinstance(value,str) or not value: raise RuntimeError(\'Required Colab secret unavailable\')\n            os.environ[key] = value\n            del value\n        with self.log.open(\'a\') as log, redirect_stdout(log), redirect_stderr(log):\n            self._prepare()\n        self.ready = True\n        self.receipt.update(status=\'READY_NOT_RUNNING\', judge_credentials_available=True,\n                            checkpoint=239, production_dataset_writes=0)\n        return self.receipt.copy()\n\n    def _prepare(self):\n        from kaggle import api\n        self.api = api; api.authenticate()\n        # Pull only code/metadata in this private cloud VM; includes private audit sidebands.\n        if api.kernels_status(KERNEL).status.name not in TERMINAL:\n            raise RuntimeError(\'Kaggle evaluation writer is active\')\n        with tempfile.TemporaryDirectory() as tmp:\n            api.kernels_pull(KERNEL+\'/18\', tmp, metadata=True)\n            notebooks = list(Path(tmp).glob(\'*.ipynb\'))\n            if len(notebooks) != 1: raise RuntimeError(\'Ambiguous supported notebook\')\n            files = supported_bundle(json.loads(notebooks[0].read_text()))\n        code = Path(\'/kaggle/working/code\'); code.mkdir(exist_ok=True)\n        stage_bundle(code, files)\n        sys.path.insert(0,str(code)); os.chdir(code)\n        from cloud.d1000 import evaluation_only as evaluation\n        from cloud.d1000.storage import info, publish\n        self.evaluation = evaluation\n        self.info = info; self.publish = publish\n        audit = json.loads((code/\'cloud/d1000/evaluation_repair_v1.json\').read_text())\n        for p,h in {**audit[\'destination_hashes\'], **audit[\'immutable_code_hashes\']}.items():\n            if sha(code/p) != h: raise RuntimeError(\'Audited source mismatch\')\n        # Private sideband hashes must match the existing repair audit.\n        for name, field in ((\'groq_probe_quota_deltas.json\',\'quota_deltas_sha256\'),\n                            (\'groq_calibration14_receipt.json\',\'groq_calibration_receipt_sha256\')):\n            if sha(code/\'cloud/d1000\'/name) != audit[field]: raise RuntimeError(\'Private audit sideband mismatch\')\n        d = info(api, STORE)\n        if not d.is_private or d.current_version_number != 239: raise RuntimeError(\'Reconcile latest checkpoint first\')\n        INPUT_ROOT.mkdir(parents=True, exist_ok=True); os.chmod(INPUT_ROOT,0o700)\n        mount = INPUT_ROOT/\'recovery239\'; mount.mkdir(exist_ok=True); os.chmod(mount,0o700)\n        for name in (\'progress.json\',\'artifacts.bin\'):\n            api.dataset_download_file(STORE+\'/239\',name,path=str(mount),quiet=True,force=True,unzip=True)\n        state = json.loads((mount/\'progress.json\').read_text())\n        if state[\'artifacts_sha256\'] != ARCHIVE_SHA or sha(mount/\'artifacts.bin\') != ARCHIVE_SHA:\n            raise RuntimeError(\'Checkpoint239 archive mismatch\')\n        # Original restore verifies latest private metadata, mounted exact progress,\n        # 912 responses, 76 questions, 3 adapters and unchanged config.\n        scope = evaluation.__dict__.copy()\n        exec(restore_adapter(files[\'cloud/d1000/evaluation_only.py\']),scope)\n        state, preservation = scope[\'restore\']()\n        if preservation[\'immutable_manifest_sha256\'] != audit[\'immutable_manifest_sha256\']:\n            raise RuntimeError(\'Frozen benchmark/model inventory mismatch\')\n        self.receipt[\'preservation\'] = preservation\n        self.receipt[\'saved_judgements\'] = verify_journals(evaluation.ROOT)\n        # Private scratch dataset carries only nonce, never research data.\n        with tempfile.TemporaryDirectory() as tmp:\n            folder = Path(tmp); (folder/\'nonce.txt\').write_text(self.session)\n            try:\n                previous = info(api,SCRATCH)\n            except Exception as exc:\n                status = getattr(getattr(exc,\'response\',None),\'status_code\',None)\n                if status != 404: raise\n                previous = None\n            if previous is not None and not previous.is_private: raise RuntimeError(\'Scratch must be private\')\n            receipt = publish(folder,SCRATCH,create=previous is None)\n            if not receipt[\'readback_verified\'] or not receipt[\'private\']: raise RuntimeError(\'Scratch readback failed\')\n            self.receipt[\'scratch_private_persistence\'] = receipt\n        if api.kernels_status(KERNEL).status.name not in TERMINAL or info(api,STORE).current_version_number != 239:\n            raise RuntimeError(\'Writer or checkpoint changed during preflight\')\n        self.state = state; self.preservation = preservation\n        # Bind only a reviewed restore adapter; original source files remain byte-identical.\n        evaluation.restore = lambda: (self.state,self.preservation)\n\n    def resume(self):\n        if not self.ready or self.started: raise RuntimeError(\'Successful preflight and one start required\')\n        api = self.api; evaluation = self.evaluation\n        if api.kernels_status(KERNEL).status.name not in TERMINAL:\n            raise RuntimeError(\'Another evaluation writer is active\')\n        if self.info(api,STORE).current_version_number != 239:\n            raise RuntimeError(\'Checkpoint changed; do not resume stale state\')\n        # A fixed private lease is created once. Existing lease => fail closed.\n        # On interrupted sessions, user must reconcile/release the lease before restart.\n        with self.log.open(\'a\') as log, redirect_stdout(log), redirect_stderr(log):\n            with tempfile.TemporaryDirectory() as tmp:\n                folder=Path(tmp); (folder/\'owner.json\').write_text(json.dumps({\'session\':self.session,\'source_checkpoint\':239}))\n                lease=self.publish(folder,LEASE,create=True)\n                self.lease_version=lease[\'version\']\n            fd=os.open(\'/kaggle/working/d1000-authoritative-writer.lock\',os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)\n            os.close(fd); self.started=True\n            self.receipt.pop(\'new_judge_calls\', None)\n            original_checkpoint=evaluation.checkpoint\n            def checked_checkpoint(state):\n                d=self.info(api,LEASE)\n                if not d.is_private or d.current_version_number != self.lease_version:\n                    raise RuntimeError(\'Authoritative writer lease changed\')\n                if api.kernels_status(KERNEL).status.name not in TERMINAL:\n                    raise RuntimeError(\'Unexpected Kaggle writer\')\n                return original_checkpoint(state)\n            evaluation.checkpoint=checked_checkpoint\n            evaluation.main()  # Original checks, calibration, replay guards and 120s persistence.\n        # Report only aggregate data after original private write/readback.\n        receipt=json.loads(Path(\'/kaggle/working/evaluation_receipt.json\').read_text())\n        self.receipt.update(status=receipt[\'status\'],checkpoint=receipt[\'last_persistence\'][\'version\'],\n            judge_counts=receipt[\'judge_counts\'],paired_judgements=receipt.get(\'paired_judgements\'),\n            readback_verified=receipt[\'last_persistence\'][\'readback_verified\'],\n            new_persisted_judgements={p:receipt[\'judge_counts\'][p]-n for p,n in ((\'groq\',36),(\'gemini\',345))})\n        return self.receipt.copy()\n'
recovery_scope = {}
exec(compile(recovery_source, '<d1000-colab-adapter-v1>', 'exec'), recovery_scope)
recovery = recovery_scope['ColabRecovery']()
print('Evaluation-only adapter loaded. No private data transferred; no judging started.')


## Preflight — zero judge calls
Restores exact private239, verifies source/archive/912 responses/3 adapters/frozen benchmark/votes/unknown503 and scratch private persistence. A newer checkpoint blocks stale restoration.


In [ ]:
# Set True only after confirming project permission for private Colab processing.
PRIVATE_COLAB_TRANSFER_APPROVED = False
try:
    result = recovery.prepare(private_colab_transfer_approved=PRIVATE_COLAB_TRANSFER_APPROVED)
    print({k:v for k,v in result.items() if k != 'preservation'})
except BaseException as exc:
    print({'status':'BLOCKED', 'stage':'preflight', 'error_type':type(exc).__name__, 'judge_calls':0})
    # Do not print exception messages, headers, credential values or private logs.


## Resume the sole evaluation writer
Requires a successful preflight. Existing lease or active Kaggle writer blocks startup. Uses unchanged research protocol and real ledgers. Do not run another notebook concurrently.


In [ ]:
# Run only after READY_NOT_RUNNING. This is the sole authoritative coordinator.
try:
    print(recovery.resume())
except BaseException as exc:
    print({'status':'BLOCKED', 'stage':'resume', 'error_type':type(exc).__name__})
    # Reconcile checkpoint/lease before restarting; never erase caches or reservations.
